# Power generation — how it works

This is the explanation and reference for `quicksat`'s power generation. It covers the panels in the power system file, how the power of each panel is worked out over a scenario run, and where the model stops.

It is not a tutorial. The worked example is `sample/power/power_generation.ipynb`. This notebook still runs, so that it cannot drift from the code. It reads `docs/power/data/`, its own copy of the inputs. The figures quoted below are true of that data.

Power builds on the scenario (`docs/explanations/decisions.md`, entry 10). The scenario gives the orbit, the sun, the eclipses, the body's orientation and the angles of the turning wings at each step. Power adds the solar panels. The consumption and the power budget have references of their own: `docs/power/power_consumption_ref.ipynb` and `docs/power/power_budget_ref.ipynb`.

In [1]:
import os
from pathlib import Path

import numpy as np

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import Q_, u
from quicksat.power.config import PowerSystem
from quicksat.power.generation import (
    SOLAR_LUMINOSITY,
    panel_geometry,
    run_generation,
)
from quicksat.scenario.config import Scenario
from quicksat.scenario.run import run_scenario
from quicksat.utils.intervals import labels_at

DATA = Path("docs") / "power" / "data"
SYSTEM_FILE = DATA / "power_system.yaml"

run = run_scenario(Scenario.from_yaml_file(DATA / "scenario.yaml"))
power_system = PowerSystem.from_yaml_file(SYSTEM_FILE)
generation_run = run_generation(power_system, run)


def with_text(extra: str = "", **replace: str) -> PowerSystem:
    """This notebook's file with some text replaced, and some lines added."""
    text = SYSTEM_FILE.read_text()
    for old, new in replace.items():
        text = text.replace(old, new)
    return PowerSystem.from_yaml_text(text + extra)

## The power system file

The power system file is one YAML file, written by hand, usually `power_system.yaml`. No notebook writes it. It is read on its own, and its panels are checked against the scenario's 3D model when the generation is worked out. The generation reads the first four keys. The other three are for the power budget, `docs/power/power_budget_ref.ipynb`.

| Key | What it holds |
| --- | --- |
| `cell_efficiency` | The cells' efficiency at their rated temperature, above 0 and at most 1, such as `30 %` or `0.30` |
| `loss_factor` | Every loss up to the array's output, as the share of the power left: the cells' temperature, ageing, mismatch between cells, UV, and the array's own wiring. Above 0 and at most 1, such as `0.80`. |
| `panels` | The panels, each under a name of your choice. Each names its part in the 3D model, or gives a normal and an area. |
| `self_shadowing` | Optional: `enabled` switches the self-shadowing on or off, and `points` sets about how many sample points each panel gets, 512 by default. Off without the block. |
| `battery` | For the budget: the capacity, the share left after fade, the maximum depth of discharge, and the starting charge |
| `conditioning` | For the budget: the efficiency of each path from the array's output to the loads and through the battery |
| `system_margin` | For the budget: the margin on all the loads together |

Each panel has a `fill_ratio`: the share of its area that the cells cover.

There is no loss per year. Set `loss_factor` for the year being sized, such as the end of life. There is no temperature model either: the efficiency and the loss factor take it in. The power after `loss_factor` is the power at the array's output. The conditioning's losses come after it, in the budget, so none is counted twice. Power keeps its own copy of the scenario file, as `decisions.md` entry 10 says, so this folder holds `scenario.yaml` and the model too.

This notebook's file:

In [2]:
print(SYSTEM_FILE.read_text())

# The reference power system of docs/power/: the solar panels, the battery and the
# power conditioning. The panels are the cells on the four wing panels, which
# turn to the sun, and the cells on the deck, which do not.
#
# A copy of the power sample's file. The reference notebooks quote figures from
# it, so it changes only together with them. They never write it.
#
# The power of a panel is the solar flux x its area x fill_ratio x
# cell_efficiency x loss_factor x the cosine of the sun's angle to the cells. It
# is scaled by the light that the Earth does not hide, and by the share of the
# panel that the rest of the model does not shade.

# The cells' efficiency at their rated temperature, such as triple-junction GaAs
# cells at about 30%.
cell_efficiency: 30 %

# Every loss up to the array's output, as the share of the power left: the cells'
# temperature, ageing, mismatch between cells, UV, and the array's own wiring.
# Set it for the year being sized, such as the end of life. 0.8

## The power of a panel

At each time of the run's grid, a panel generates

$$P = \Phi \, A \, f \, \eta \, k \, \max(\cos\theta, 0) \, L \, \ell$$

- $\Phi$ is the solar flux at the satellite: the sun's luminosity spread over a sphere through the satellite, $L_\odot / (4 \pi d^2)$. $d$ is the distance from the satellite to the sun, and $L_\odot$ is astropy's value, the IAU 2015 nominal $3.828 \times 10^{26}$ W. That gives 1361 W/m² at 1 au. Over a year, the flux runs from about 1408 W/m² in early January to 1317 W/m² in early July, ±3.3%.
- $A$ is the panel's area, $f$ its fill ratio, $\eta$ the cell efficiency, and $k$ the loss factor. The result is the power at the array's output.
- $\theta$ is the angle between the panel's normal and the sun, in body axes. The body's orientation comes from the run, through its slews. Only the side the cells face generates, so a panel with the sun behind it gives nothing.
- $L$ is the light from the run: the fraction of the sun's disk that the Earth does not hide. It is 1 in sunlight and 0 in the umbra, and falls through the penumbra between them.
- $\ell$ is the panel's lit fraction from the self-shadowing: the share of its area that no other part of the model shades. It is 1 without the self-shadowing.

Light reflected by the Earth and the Earth's infrared are left out.

This notebook's run starts on 1 October, when the Earth is close to its mean distance from the sun:

In [3]:
at_1_au = (SOLAR_LUMINOSITY / (4 * np.pi * Q_(1, "au") ** 2)).to(u.W / u.m**2)
flux = generation_run.flux
print(f"flux at 1 au     {at_1_au:.1f}")
print(f"flux in the run  {flux.min():.1f} to {flux.max():.1f}")

flux at 1 au     1361.2 W / m2
flux in the run  1357.8 W / m2 to 1358.1 W / m2


## Panels from the 3D model

A panel that names a `part` takes its shape from the scenario's 3D model, the GLB file that `3d_model` names. quicksat reads the model's triangles, and places each node's triangles in body axes through the transforms of the node and all the nodes above it.

- **The area** is the sum of the part's triangle areas.
- **The normal** is the area-weighted mean of the triangle normals. A triangle's normal points to the side from which its corners run counter-clockwise, as glTF draws its front face.
- **The part must be flat.** Every triangle must face within 1° of the mean normal. A closed shape, such as a wing modelled as a thin box, faces no one way and is refused. So the cells must be a part of their own, as this notebook's model has them: `Wing +y inner cells` is the face of cells, and `Wing +y inner` the panel's box.
- **The name must match exactly one node** of the model's scene, and that node must hold triangles.

The geometry is then not typed in twice, and the self-shadowing can use the same triangles. `docs/explanations/decisions.md`, entry 11, gives the reasons. The panels of this notebook's file:

In [4]:
generation_run.panel_table.style.hide(axis="index").format(
    {"area [m2]": "{:.4f}", "fill ratio [%]": "{:.0f}"}
)

panel,part,area [m2],fill ratio [%],normal,turns with
wing +y inner,Wing +y inner cells,0.4026,85,"[+0.000, +0.000, -1.000]",wing +y
wing +y outer,Wing +y outer cells,0.4026,85,"[+0.000, +0.000, -1.000]",wing +y
wing -y inner,Wing -y inner cells,0.4026,85,"[+0.000, +0.000, -1.000]",wing -y
wing -y outer,Wing -y outer cells,0.4026,85,"[+0.000, +0.000, -1.000]",wing -y
deck,Deck cells,0.0716,80,"[+0.000, +0.000, -1.000]",


A part that is not a flat panel stops the generation, with what is wrong. The bus is a box:

In [5]:
bus = with_text(**{"part: Deck cells": "part: Bus"})
try:
    panel_geometry(bus, run.scenario)
except ValueError as error:
    print(error)

panel 'deck': the part 'Bus' faces no one way, like a closed box: it is not a flat panel. Name the face that holds the cells as a part of its own.


### Panels that turn

A panel whose part sits under an articulation's node turns with it, such as a panel of a solar wing. quicksat finds the articulation from the model's node tree, so the power system file does not repeat it. At each step, the panel's normal turns about the drive axis by the articulation's angle from the run. The scenario reference, `docs/scenario/scenario_ref.ipynb`, explains how the angle is set.

While a wing tracks the sun inside its range, its cells meet the sun at the cosine $\sqrt{1 - (\hat{s} \cdot \hat{a})^2}$, where $\hat{a}$ is the drive axis. In a mode that parks the wing, the cells keep the direction they have at the park angle. Here the wings turn about y and park at 0° in imaging, where the cells face -z, as the deck cells always do:

In [6]:
sun = run.sun_body
modes = labels_at(run.mode, run.times)
# the run is open at its end, so the last time takes the last activity's mode
modes[-1] = run.occurrences[-1].activity.mode
tracking = modes != "imaging"
wing = generation_run.cosines["wing +y outer"]

print(
    "wing +y outer, tracking: largest difference from sqrt(1 - (s.y)^2) "
    f"{np.abs(wing - np.sqrt(1 - sun[:, 1] ** 2))[tracking].max():.1e}"
)
print(
    "wing +y outer, parked:   largest difference from -s.z "
    f"{np.abs(wing + sun[:, 2])[~tracking].max():.1e}"
)
print(
    "deck, always:            largest difference from -s.z "
    f"{np.abs(generation_run.cosines['deck'] + sun[:, 2]).max():.1e}"
)

wing +y outer, tracking: largest difference from sqrt(1 - (s.y)^2) 6.7e-16
wing +y outer, parked:   largest difference from -s.z 0.0e+00
deck, always:            largest difference from -s.z 0.0e+00


A panel can turn with one articulation only. A part under two is refused.

## Typed-in panels

A panel can give its `normal` and its `area` in place of a `part`. The normal is a signed body axis, such as `+x`, or three numbers in body axes, which quicksat scales to unit length. Typed-in panels suit a scenario with no 3D model, or a model with no clear panels. They also let a new panel be tried without editing the model.

A typed-in panel has no shape. So it stays fixed in body axes, is never shaded, and shades nothing. Whenever one is used, the run gives a note, as text for the output rather than a Python warning:

In [7]:
side = "  side +x: {normal: +x, area: 0.2 m2, fill_ratio: 80 %}\n"
typed = with_text(**{"self_shadowing:": side + "self_shadowing:"})
for note in run_generation(typed, run).notes:
    print(note)

Some panels are typed in, not taken from the 3D model: side +x. They stay fixed in body axes, are never shaded, and shade nothing. The others come from the model.


With no 3D model, every panel must be typed in, and the note says so. If the self-shadowing is on but no panel comes from the model, a second note says that nothing is shaded.

## The orbits and the energy

The tables count whole orbits from the start of the run, one orbital period each. The period is the nodal period for a TLE or an SSO, and the measured one for a trajectory file. A cut orbit at the end of the run is left out, and a run shorter than one orbit has no rows. A part of an orbit would skew the means: in the Orekit case below, averaging over a whole day rather than whole orbits moves a panel's mean by up to 3.4%.

- **The energy** treats the power as changing in a straight line between the steps of the grid. The energy is exact for that line, also where an orbit starts or ends between two steps.
- **The mean power** is the orbit's energy over its period.
- **The sunlit mean** is the energy generated outside the eclipse, over the time outside it. The penumbra counts as eclipse, as in the scenario. What the panels generate in the penumbra still counts in the orbit's energy.

`orbit_table` gives the panels together, and `panel_energy_table` each panel's energy:

In [8]:
generation_run.orbit_table.style.hide(axis="index").format(
    {
        "eclipse [min]": "{:.2f}",
        "mean power [W]": "{:.1f}",
        "sunlit mean power [W]": "{:.1f}",
        "energy [Wh]": "{:.1f}",
    }
)

orbit,start [UTC],eclipse [min],mean power [W],sunlit mean power [W],energy [Wh]
1,2026-10-01T09:00:00.000,34.32,275.5,430.6,435.5
2,2026-10-01T10:34:51.055,34.32,275.1,429.9,434.8
3,2026-10-01T12:09:42.109,34.32,273.1,426.8,431.8


## Self-shadowing

A part of the spacecraft that lies between a panel and the sun shades it. The cosine alone cannot show this. On a plain box the cosine is exact, because a convex body cannot shade its own faces. Shading needs parts that stand out, such as wings, a payload housing or an antenna. In this notebook's model, the payload housing on the deck shades the deck cells and the inner panel of one wing.

**The method.** Each panel from the model gets sample points on its cells. Each triangle of the panel is cut into $n^2$ equal smaller ones, with $n$ set by its share of the area, and the points are their centres. At each step, a ray goes from each point toward the sun, in body axes. A point is shaded when a triangle of another part blocks its ray. All rays at a step share the sun's direction. So the model is projected along that direction onto a plane, once a step, and a point is shaded when it falls inside a projected triangle that lies further toward the sun. This is the Möller-Trumbore test, arranged for one direction. It needs no extra package.

- **Turning parts move.** At each step, the parts under an articulation take its angle, so a wing both casts its shadow and catches others' where its angle puts it.
- **Only where it matters.** No ray is cast while a panel faces away from the sun, or in the umbra, since the power is zero there anyway. A triangle that lies wholly behind the panel's points, or whose projection misses theirs, is left out of the test.
- **Self-hits.** A ray can hit the surface it starts on, through rounding, and read a panel in full sun as partly shaded. Each ray starts 1 µm off the cells, along their normal, which removed every self-hit in the prototype, down to 0.001° above the panel's plane.
- **Late shadow edges.** A shadow first touches a panel at a corner, and its area grows slowly from there, so the first point is covered late. The time a panel is shaded counts only where more than 1% of it is shaded. In the prototype, the shaded times then ended within one step of the exact ones, from 512 points up.
- **No strings.** A shaded area costs only its share of the panel's power. A real string of cells loses more, since its weakest cell limits it.

The prototype checked the lit fraction against exact shadow areas, over a day of the scenario sample with the original model:

| Points a panel | Largest error | RMS error | Energy error |
| --- | --- | --- | --- |
| 32 | 0.039 | 0.0044 | 0.011% |
| 128 | 0.022 | 0.0018 | 0.006% |
| 512 | 0.012 | 0.0008 | 0.007% |
| 2048 | 0.006 | 0.0004 | 0.002% |

`shading_table` sums up what the shading costs each panel from the model, over the run:

In [9]:
generation_run.shading_table.style.hide(axis="index").format(
    {
        "lowest lit fraction": "{:.3f}",
        "shaded [min]": "{:.1f}",
        "energy lost [Wh]": "{:.2f}",
        "energy lost [%]": "{:.2f}",
    }
)

panel,lowest lit fraction,shaded [min],energy lost [Wh],energy lost [%]
wing +y inner,0.398,31.3,8.49,2.68
wing +y outer,1.000,0.0,0.00,0.00
wing -y inner,1.000,0.0,0.00,0.00
wing -y outer,1.000,0.0,0.00,0.00
deck,0.214,38.2,3.68,8.17


More points give a finer lit fraction and take longer. The energy lost hardly changes past a few hundred points, which is why 512 is the default:

In [10]:
for points in (32, 128, 512, 2048):
    finer = run_generation(with_text(**{"points: 512": f"points: {points}"}), run)
    lost = finer.shading_table.set_index("panel")["energy lost [Wh]"]
    print(
        f"{points:5d} points: energy lost {lost['wing +y inner']:.2f} Wh on wing +y "
        f"inner, {lost['deck']:.2f} Wh on the deck"
    )

   32 points: energy lost 8.48 Wh on wing +y inner, 3.74 Wh on the deck


  128 points: energy lost 8.41 Wh on wing +y inner, 3.69 Wh on the deck


  512 points: energy lost 8.49 Wh on wing +y inner, 3.68 Wh on the deck


 2048 points: energy lost 8.49 Wh on wing +y inner, 3.68 Wh on the deck


## Checked against Orekit

`tests/power/test_generation.py` checks the generation against Orekit, on a case built for it. It has six 1 m² faces, typed in, along the velocity, the orbit normal and the nadir, both ways. It flies a 510 km sun-synchronous orbit at 10:30 LTAN, in nadir pointing, for one day at 10 s. Orekit ran its own SGP4, frames, sun and attitude, so the reference does not share quicksat's astropy code. `tests/power/data/README.md` describes the case.

| Quantity | quicksat | Allowed |
| --- | --- | --- |
| Each face's cosine, at each step | 1.0 × 10⁻⁴ | 2 × 10⁻⁴ |
| The sun's direction in body axes | 16 to 27 arcsec | 30 arcsec |
| The flux | 8.7 × 10⁻⁵ | 10⁻⁴ |
| Each face's mean power over 14 whole orbits | 0.024% | 0.1% |

- **The sun's direction.** Orekit's sun is the geometric one, seen from the centre of the Earth. quicksat's is astropy's apparent sun, which adds 20.5 arcsec of aberration, seen from the satellite, which adds up to 9.5 arcsec of parallax. The same distance accounts for the flux.
- **The shadow.** Orekit's reference uses a cylindrical shadow. quicksat's shadow has a penumbra on WGS84, which moves each eclipse edge by seconds. So the test takes the means with Orekit's lit flag, which checks the geometry alone. The shadow is checked against Orekit's eclipse times in `tests/orbit/test_geometry.py`.

## Limitations

What the power generation does not do, yet or at all:

- **No light from the Earth.** Sunlight reflected by the Earth and the Earth's infrared are left out. They matter most for panels that face the Earth.
- **No temperature model, and no ageing per year.** The cell efficiency and the loss factor take them in, set for the year being sized.
- **No strings of cells.** A shaded area costs only its share of the panel's power.
- **Typed-in panels have no shape.** They stay fixed in body axes, are never shaded, and shade nothing.
- **One articulation a panel.** A panel turns with one articulation at most, and its wing turns at once, with no drive and no rate limit, as the scenario sets it.
- **A steady sun.** The flux comes from the sun's nominal luminosity and its distance. The solar cycle's changes, about 0.1%, are left out.
- **Short runs.** A run covers an orbit to a day or two, so the beta angle hardly changes in it. Seasons need a scenario each, such as winter and summer.